# Minimal GRPO Test — Qwen3.5-0.8B

Quick test to see if GRPO training works at all with Qwen3.5-0.8B.
Tries `FastLanguageModel` first (avoids VLM code path).

In [ ]:
%%capture
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["UNSLOTH_RETURN_HIDDEN_STATES"] = "1"

import importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
!pip uninstall -y torchcodec

import torch
torch._dynamo.config.disable = True
torch._dynamo.config.suppress_errors = True

In [ ]:
# Neuter torch.compile
import torch
_real_compile = torch.compile
def _fake_compile(fn=None, *args, **kwargs):
    if fn is None:
        return lambda f: f
    return fn
torch.compile = _fake_compile

import shutil
shutil.rmtree("/content/unsloth_compiled_cache", ignore_errors=True)
print("torch.compile neutered, compiled cache deleted")

## Test 1: FastLanguageModel

In [ ]:
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3.5-0.8B",
    max_seq_length=2048,
    load_in_4bit=True,
    fast_inference=False,
)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=32, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth",
)
print(f"Model type: {type(model)}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
dataset = Dataset.from_list([
    {"prompt": "Extract the dollar amounts from this text: TOTAL $12.50"},
] * 8)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[lambda completions, **kw: [1.0] * len(completions)],
    args=GRPOConfig(
        output_dir="/tmp/test",
        max_steps=1,
        num_generations=4,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        max_completion_length=128,
        max_prompt_length=256,
        report_to="none",
    ),
    train_dataset=dataset,
)
print("Trainer created")

In [ ]:
trainer.train()
print("\n=== Test 1 PASSED: FastLanguageModel + GRPO works ===")

## Test 2: FastLanguageModel + SFT LoRA + Real Data Format

If Test 1 passes, test with our actual data format (system + user messages, DSPy reward functions).

In [ ]:
# Clean up Test 1
del model, tokenizer, trainer, dataset
import gc; gc.collect()
torch.cuda.empty_cache()
print("Cleaned up")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/qwen35_08b_finetuning"
SFT_LORA_PATH = f"{BASE_DIR}/qwen35-08b-dspy-format-lora"
DATA_PATH = f"{BASE_DIR}/grpo_extractor_mutated.jsonl"

In [ ]:
# Load SFT LoRA — try FastLanguageModel first
model2, tokenizer2 = FastLanguageModel.from_pretrained(
    model_name=SFT_LORA_PATH,
    max_seq_length=4096,
    load_in_4bit=False,
    fast_inference=False,
    use_gradient_checkpointing="unsloth",
)
print(f"Model type: {type(model2)}")
print(f"Trainable params: {sum(p.numel() for p in model2.parameters() if p.requires_grad):,}")

In [ ]:
import json

# Load first 16 examples as test data
with open(DATA_PATH) as f:
    raw = [json.loads(line) for i, line in enumerate(f) if i < 16]

# For FastLanguageModel, prompts use plain strings (no VLM typed blocks)
test_data = []
for r in raw:
    test_data.append({
        "prompt": r["prompt"],  # Already [{"role": "system", "content": "..."}, {"role": "user", "content": "..."}]
        "ground_truth_ids": json.dumps(r["ground_truth_ids"]),
        "ground_truth_values": json.dumps(r["ground_truth_values"]),
    })

test_dataset = Dataset.from_list(test_data)
print(f"Test dataset: {len(test_dataset)} examples")
print(f"Sample prompt roles: {[m['role'] for m in test_dataset[0]['prompt']]}")

In [ ]:
import ast

def parse_list(s):
    s = s.strip()
    try: return json.loads(s)
    except: pass
    try:
        r = ast.literal_eval(s)
        return r if isinstance(r, list) else []
    except: return []

def parse_dspy(text):
    try:
        if "[[ ## field_ids ## ]]" not in text: return [], []
        ids = text.split("[[ ## field_ids ## ]]")[1].split("[[ ## field_values ## ]]")[0].strip()
        vals = text.split("[[ ## field_values ## ]]")[1]
        if "[[ ## completed ## ]]" in vals: vals = vals.split("[[ ## completed ## ]]")[0].strip()
        else: vals = vals.strip()
        return parse_list(ids), parse_list(vals)
    except: return [], []

def get_text(c):
    if isinstance(c, str): return c
    if isinstance(c, list) and c:
        msg = c[0]
        if isinstance(msg, dict):
            content = msg.get("content", "")
            if isinstance(content, str): return content
            if isinstance(content, list): return " ".join(x.get("text","") for x in content if isinstance(x,dict))
    return str(c)

def format_reward(completions, **kw):
    scores = []
    for c in completions:
        t = get_text(c)
        ids, vals = parse_dspy(t)
        s = 0.0
        if all(m in t for m in ["[[ ## field_ids ## ]]", "[[ ## field_values ## ]]", "[[ ## completed ## ]]"]):
            s += 1.0
            if ids and len(ids) == len(vals): s += 1.0
        scores.append(s)
    return scores

def accuracy_reward(completions, ground_truth_ids, ground_truth_values, **kw):
    scores = []
    for c, gt_i, gt_v in zip(completions, ground_truth_ids, ground_truth_values):
        t = get_text(c)
        gt_ids, gt_vals = json.loads(gt_i), json.loads(gt_v)
        pred_ids, pred_vals = parse_dspy(t)
        if not gt_ids: scores.append(1.5 if not pred_ids else 0.0); continue
        if not pred_ids: scores.append(0.0); continue
        gt_s, pred_s = set(gt_ids), set(pred_ids)
        tp = len(gt_s & pred_s)
        p = tp/len(pred_s) if pred_s else 0
        r = tp/len(gt_s) if gt_s else 0
        f1 = 2*p*r/(p+r) if p+r else 0
        gt_map, pred_map = dict(zip(gt_ids,gt_vals)), dict(zip(pred_ids,pred_vals))
        cv = sum(1 for fid in gt_s&pred_s if str(pred_map.get(fid,"")).strip()==str(gt_map[fid]).strip())
        scores.append(f1*1.5 + (cv/len(gt_s))*1.5)
    if completions:
        gt = json.loads(ground_truth_ids[0])
        pred, _ = parse_dspy(get_text(completions[0]))
        print(f"--- GT: {gt} | Pred: {pred} | Score: {scores[0]:.2f}")
    return scores

print("Reward functions defined")

In [ ]:
trainer2 = GRPOTrainer(
    model=model2,
    processing_class=tokenizer2,
    reward_funcs=[format_reward, accuracy_reward],
    args=GRPOConfig(
        output_dir="/tmp/test2",
        max_steps=2,
        num_generations=4,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        max_completion_length=256,
        max_prompt_length=4096,
        learning_rate=5e-6,
        report_to="none",
    ),
    train_dataset=test_dataset,
)
print("Trainer 2 created")

In [ ]:
# Patch rope_deltas just in case
import types
_orig = trainer2.training_step.__func__
def _patched(self, model, inputs, *args, **kwargs):
    for m in self.accelerator.unwrap_model(model).modules():
        if hasattr(m, "rope_deltas"): m.rope_deltas = None
    return _orig(self, model, inputs, *args, **kwargs)
trainer2.training_step = types.MethodType(_patched, trainer2)
print("rope_deltas patch applied")

In [ ]:
trainer2.train()
print("\n=== Test 2 PASSED: SFT LoRA + real data + reward functions ===")